# Detrending Moving Average (DMA) Test

In this notebook we implement and experiment with the DMA test proposed by Sikora (See https://www.sciencedirect.com/science/article/pii/S0960077918309421) for identifying $fBm_H$.

Many methods for inferring the Hurst parameter have been established. 
However there are very few tests for determining if a given process is 
$fBm_H$ or not. Thus supposed our null hyothesis is: 
$$\mathcal{H}_0: (X_t)_{tgeq 0} \quad \text{is a } fBm_H$$
The DMA test is (at time of writing, and according to 
Sikora 2018) one of only two such methods.

The DMA statistic is defined for empirical observations 
$X = \{X(1), \dots, X(N)\}$ and window parameter $n$ by:

$$\sigma^2(n) = \frac{1}{N-n}\sum_{j=n}^{N}(X(j) - \tilde{X}_n(j))^2$$

where $\tilde{X}_n(j) = \frac{1}{n}\sum_{k=0}^{n-1} X(j-k)$ is the 
rolling $n$-window average. The statistic measures how much the 
process deviates from its own local trend, and importantly scales as 
$\sigma^2(n) \sim C_H n^{2H}$, encoding the Hurst parameter $H$.

We can show that 
$(N-n)\sigma^2(n)$ follows a generalised chi-squared distribution:

$$(N-n)\sigma^2(n) \stackrel{d}{=} \sum_{j=1}^{N-n+1} \lambda_j(n) U_j, \quad U_j \sim \chi^2(1)$$

where $\lambda_j(n)$ are eigenvalues of the covariance matrix 
$\tilde{\Sigma}$ (of a related proccess Y) under $H_0$, and the $\chi^2(1)$ variables arise 
simply because $Z \sim \mathcal{N}(0,1) \implies Z^2 \sim \chi^2(1)$.

The p-value is then computed via Monte Carlo: we simulate $L$ draws 
from this null distribution and count what proportion are more extreme 
than our observed $t = \sigma^2(n)$, giving:

$$p = \frac{2\min\{\#\{\sigma^2_l(n) \leq t\},\ \#\{\sigma^2_l(n) > t\}\}}{L}$$





In [ ]:
import numpy as np

#Define moving average function tidle{X}
def moving_avg(X, n, j):
    return np.mean(X[j - n:j])

#print(moving_avg(np.array([1, 2, 3, 4, 5]), 3, 4)) # should return 3

#fBm covariance function
def cov_fbm(j, k, H):
    """Covariance of fBm: E[B_H(j) B_H(k)] = 0.5*(|j|^2H + |k|^2H - |j-k|^2H)"""
    return 0.5 * (abs(j)**(2*H) + abs(k)**(2*H) - abs(j-k)**(2*H))


#Define the DMA test statistic function (step 1 from paper page 3)
def DMA_test(X, n, H):
    """
    Perform the Detrending Moving Average (DMA) test on a given trajectory.

    Parameters:
    X (numpy.ndarray): The input trajectory to be analyzed.
    n (int): The number of data points in the trajectory.
    n (int): The window size for the moving average.
    H (float): The Hurst exponent.

    Returns:
    float: The DMA value and eigenvalues for the given trajectory.
    """
    
    N = len(X)
    #step 1 compute DMA test statistic 
    t_DMA = 1/(N-n) * np.sum([(X[j] - moving_avg(X, n, j))**2 for j in range(n, N)])

    return t_DMA

#DMA_test(np.array([1, 2, 3, 4, 5]), 3, 0.7) # should return a float value

#step 2:Covariance eigenvalues function for DMA test statistic
def cov_matrix_eigenvals(N, n, H):
    M = N - n + 1
    sigma_tilde = np.zeros((M, M))
    for j in range(M):
        for k in range(M):
            term1 = (1 - 1/n)**2 * cov_fbm(j + n, k + n, H)
            
            sum_jk = sum(cov_fbm(j + n, k + m, H) for m in range(1, n))  
            sum_kj = sum(cov_fbm(k + n, j + m, H) for m in range(1, n)) 
            term2 = (1/n**2 - 1/n) * (sum_jk + sum_kj)
            
            term3 = (1/n**2) * sum(cov_fbm(j + l, k + m, H) for l in range(1, n) for m in range(1, n))
            
            sigma_tilde[j, k] = term1 + term2 + term3
    return np.linalg.eigvalsh(sigma_tilde)



#Get p-value using Monte carlo simulation to compute the theoretical distribution of the DMA test statistic (steps 3,4,5 from paper)
def monte_carlo_DMA(X, n, H, L):
    """
    Perform a Monte Carlo simulation to compute the distribution of the DMA test statistic.

    Parameters:
    X (numpy.ndarray): The input trajectory to be analyzed.
    n (int): Moving average window size.
    H (float): The Hurst exponent.
    L (int): The number of Monte Carlo simulations to perform (L in the paper).

    Returns:
    float: p-value
    """
    
    N = len(X)
    
    #Step 3: L times generate a sample U^l = {U^l_1, ..., U^l_{N-n+1}} of size N-n+1 from chi-squared distribution with 1 degree of freedom
    U = np.random.chisquare(df=1, size=(L, N-n+1))
    
    #step 4` compute the DMA distribution using each sample U^l
    DMA_distribution = np.zeros(L)
    lambdas = cov_matrix_eigenvals(N, n, H)
    for l in range(L):
        DMA_distribution[l] = 1/(N) * np.sum([lambdas * U[l, :]])

    #step 5: compute the doubled-tailed p-
    count_big = np.sum(DMA_distribution > DMA_test(X, n, H))
    count_small = np.sum(DMA_distribution <= DMA_test(X, n, H))
    p = 2* max(count_big, count_small) / L  # multiply by 2 for two-tailed test

    return p



#monte_carlo_DMA(np.array([1, 2, 3, 4, 5]), 3, 0.7, 1000) # should return a float value between 0 and 1



## Reproducing Paper's Results

We will check our above test using $H_real = 0.25$